# UTA-RLDD participant-held-out cloud evaluation

This notebook processes UTA-RLDD videos in Kaggle storage. It uses the five published participant folds, keeps participants disjoint, trains a three-class temporal GRU on 30-second feature sequences, calibrates each fold's drowsiness-warning threshold on inner validation participants, and reports test results by held-out participant fold.

The labels are participant-provided predominant states for whole videos, not event timestamps. This notebook can measure missed drowsy videos and false warning videos, but cannot measure true event-level misses or delay from event onset. The Kaggle inputs combine folds 1–4 from `rishab260/uta-reallife-drowsiness-dataset` and fold 5 from `mathiasviborg/uta-rldd-fold5`. Both uploaders list CC0, but the mirrors are unofficial and provenance/rights are not independently verified. Cite the UTA-RLDD authors and do not publish identifiable participant imagery.

This is a research prototype, not a safety-certified warning system.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe>=0.10.9,<0.11'], check=True)
import mediapipe as mp
print('MediaPipe:', mp.__version__)

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), 'Select a GPU accelerator in Kaggle notebook settings.'
print('GPU:', torch.cuda.get_device_name(0))
REPO = '/kaggle/working/SafeDrive-Monitor'
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Pravith101/SafeDrive-Monitor.git', REPO], check=True)
%cd /kaggle/working/SafeDrive-Monitor

In [ ]:
from pathlib import Path
input_root = Path('/kaggle/input')
videos = [p for p in input_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.mp4', '.mov', '.m4v'} and p.stem.split('_')[0] in {'0', '5', '10'}]
print('UTA-RLDD videos discovered:', len(videos))
assert videos, 'Attach rishab260/uta-reallife-drowsiness-dataset under Add Input.'
print('Sample paths:', *[str(p) for p in videos[:5]], sep='\n')

In [ ]:
import subprocess
subprocess.run([
    'python', 'models/uta_rldd_cloud.py',
    '--dataset-dir', '/kaggle/input',
    '--output-dir', '/kaggle/working/uta-rldd-results',
    '--cache', '/kaggle/working/uta-rldd-features.npz',
    '--epochs', '12',
    '--max-false-warning-video-rate', '0.10',
], check=True)

In [ ]:
import json
report_path = Path('/kaggle/working/uta-rldd-results/uta_rldd_cv_report.json')
report = json.loads(report_path.read_text())
print(json.dumps({key: report[key] for key in ('participant_count', 'video_count', 'mean_test_video_accuracy', 'mean_test_video_macro_f1', 'mean_drowsy_video_recall', 'mean_non_drowsy_video_false_warning_rate', 'mean_missed_drowsy_videos', 'mean_median_first_warning_seconds_after_clip_start')}, indent=2))
print('Saved report:', report_path)
print('Saved fold checkpoints:', len(list(Path('/kaggle/working/uta-rldd-results').glob('uta_rldd_fold_*.pth'))))